# H-NEG × Gemma 4 — Reproducible Walkthrough

**Deterministic Loop Breaking: A Hierarchical Neuro-Symbolic State-Machine Architecture for Graph-Based Code Comprehension in Gemma 4 Agents**

This notebook walks through the H-NEG runtime:

1. Build a Code Property Graph (CPG) and Context Integrity Contract (`SR_id`)
2. Loop 1 — reflex validation (< 50 ms)
3. Macro layer — dynamic pushdown statechart with `GRACEFUL_HALT`
4. Micro layer — anti-loop failure trie and identifier-scope logit masking
5. End-to-end governed generation with Gemma 4 (GPU required)

Sections 1–4 run on CPU. Section 5 needs a GPU and access to the Gemma 4 weights (on Kaggle: *Add Input → Models → Gemma 4*).

In [ ]:
# Kaggle: clone the repo (replace with your public GitHub URL) and install deps
# !git clone https://github.com/<your-user>/hneg-gemma4-agent.git
# %cd hneg-gemma4-agent
# !pip install -q -r requirements.txt
import os, sys
sys.path.insert(0, os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))

In [ ]:
from src import (CodePropertyGraph, reflex_check, DynamicStateManager, Phase,
                 FailedDiffTrie, HNEGLogitsProcessor, IdentifierScopeGate, load_config)
from src.state_machine import Event

ROOT = os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else ".")
cfg = load_config(os.path.join(ROOT, "hneg_config.yaml"))
cfg["governance"], cfg["symbolic_engine"]

## 1. Code Property Graph & Context Integrity Contract

In [ ]:
import tempfile, pathlib, textwrap

demo_repo = pathlib.Path(tempfile.mkdtemp()) / "demo"
(demo_repo / "auth").mkdir(parents=True)
(demo_repo / "auth" / "jwt.py").write_text(textwrap.dedent('''
    import hmac
    import hashlib
    from auth.keys import load_secret

    def sign(payload: bytes) -> str:
        return hmac.new(load_secret(), payload, hashlib.sha256).hexdigest()

    def verify_token(payload: bytes, signature: str) -> bool:
        expected = sign(payload)
        return expected == signature   # BUG: timing-unsafe comparison
'''))
(demo_repo / "auth" / "keys.py").write_text("def load_secret() -> bytes:\n    return b'dev-secret'\n")

cpg = CodePropertyGraph.from_repo(demo_repo)
print("SR_id:", cpg.sr_id)
print(cpg.summary())
node = cpg.find_definitions("verify_token")[0]
print("Active AST node:", node.node_id)
print("Callees:", [c.node_id for c in cpg.callees(node.node_id)])
allow = cpg.symbol_allowlist("auth/jwt.py", "verify_token")
sorted(allow)

## 2. Loop 1 — Reflex check

In [ ]:
good = '''
def verify_token(payload: bytes, signature: str) -> bool:
    expected = sign(payload)
    return hmac.compare_digest(expected, signature)
'''
bad_syntax = "def verify_token(payload, signature)\n    return True\n"
out_of_scope = good.replace("hmac.compare_digest", "secure_compare")

for name, src in [("good", good), ("bad_syntax", bad_syntax), ("out_of_scope", out_of_scope)]:
    r = reflex_check(src, allowlist=allow, latency_cap_ms=cfg["symbolic_engine"]["reflex_latency_cap_ms"])
    print(f"{name:13s} ok={r.ok!s:5s} {r.latency_ms:6.2f} ms  {r.error or ''}")

## 3. Macro layer — pushdown statechart and `GRACEFUL_HALT`

In [ ]:
sm = DynamicStateManager.from_config(cfg)
sm.start("auth/jwt.py", tuple(allow))
sm.fire(Event.TARGET_LOCATED, ast_node=node.node_id)
print("edit:      ", sm.current)

# Sub-task: a dependency is missing -> push a child frame
sm.fire(Event.MISSING_DEP, file_scope="auth/keys.py", ast_node="load_secret")
print("push:      ", sm.current, "| stack depth", len(sm.stack))
sm.fire(Event.DEP_RESOLVED)
print("pop:       ", sm.current, "| stack depth", len(sm.stack))

print("emit_structured_patch allowed?", sm.tool_allowed("emit_structured_patch"))
print("run_test_suite allowed?       ", sm.tool_allowed("run_test_suite"))

# Three failed builds -> deterministic halt
for _ in range(3):
    if sm.current.phase == Phase.REPO_ANALYSIS:
        sm.fire(Event.TARGET_LOCATED)
    sm.fire(Event.PATCH_EMITTED)
    sm.fire(Event.BUILD_FAILED, payload={"error": "AssertionError"})
print("final:     ", sm.current.phase, "k =", sm.k)
[(e.event, e.source, e.target, e.k) for e in sm.trace]

## 4. Micro layer — anti-loop failure trie

In [ ]:
trie = FailedDiffTrie(min_prefix_len=3)
failed_patch = [101, 7, 42, 9, 13, 88]      # token ids of a patch that failed tests
trie.insert(failed_patch)

for prefix in ([101, 7], [101, 7, 42], [101, 7, 42, 9], [101, 7, 55]):
    print(prefix, "-> blocked next:", trie.blocked_next(prefix))

In [ ]:
import torch

class ToyTokenizer:
    eos_token_id = 0
    def decode(self, ids, skip_special_tokens=True):
        return ""

proc = HNEGLogitsProcessor(ToyTokenizer(), failed_trie=trie, patch_start=0)
scores = torch.zeros(1, 128)
masked = proc(torch.tensor([[101, 7, 42]]), scores)
print("token 9 logit after mask:", masked[0, 9].item())     # -inf: replicating the failed patch is impossible
print("token 10 logit after mask:", masked[0, 10].item())

## 4b. Sandbox: Gemma 4 E2B GGUF via llama.cpp (one T4, 8–12 GB GPU, or CPU)

Runs the same governance stack on `unsloth/gemma-4-E2B-it-GGUF` / `gemma-4-E2B-it-Q4_K_M.gguf` (3.1 GB; pass `--gguf-repo unsloth/gemma-4-E4B-it-GGUF --gguf-file gemma-4-E4B-it-Q4_K_M.gguf` for E4B) and
checks the gate, the anti-loop trie, and `GRACEFUL_HALT` with the real tokenizer. Set `RUN_SANDBOX = True` to
download and run it (`pip install llama-cpp-python huggingface_hub`; for a GPU, build with `CMAKE_ARGS="-DGGML_CUDA=on"`).

In [ ]:
RUN_SANDBOX = False
if RUN_SANDBOX:
    import subprocess
    gpu_layers = "-1" if torch.cuda.is_available() else "0"
    print(subprocess.run([sys.executable, os.path.join(ROOT, "experiments", "sandbox_smoke.py"),
                          "--n-gpu-layers", gpu_layers], capture_output=True, text=True).stdout)

## 5. Governed generation with Gemma 4 (GPU)

Loads Gemma 4, attaches `HNEGLogitsProcessor` with the CPG identifier gate and the failure trie,
and runs the bounded convergence loop on the demo bug. Hugging Face Gemma checkpoints apply the
final-logit soft-cap inside the model, so `logit_softcap` stays `null` in the config.

In [ ]:
RUN_GEMMA = torch.cuda.is_available()
MODEL_ID = cfg["agent"]["model_id"]   # on Kaggle, point this at the attached model path
print("GPU available:", RUN_GEMMA, "| model:", MODEL_ID)

In [ ]:
if RUN_GEMMA:
    import re
    from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessorList

    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype="auto").eval()

    gate = IdentifierScopeGate(tok, allow)
    trie = FailedDiffTrie(cfg["symbolic_engine"]["trie_min_prefix_len"])
    proc = HNEGLogitsProcessor(tok, failed_trie=trie, grammar_gate=gate)

    source = (demo_repo / "auth" / "jwt.py").read_text()
    messages = [{"role": "user", "content":
                 "Fix the timing-unsafe comparison in verify_token. Return the full corrected file "
                 f"in one ```python block.\n\n```python\n{source}```"}]

    sm = DynamicStateManager.from_config(cfg)
    sm.start("auth/jwt.py", tuple(allow))
    while not sm.done:
        if sm.current.phase == Phase.REPO_ANALYSIS:
            sm.fire(Event.TARGET_LOCATED, ast_node=node.node_id)
        ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
        proc.reset(ids.shape[1])
        out = model.generate(ids, max_new_tokens=512, do_sample=False,
                             logits_processor=LogitsProcessorList([proc]))
        new_ids = out[0, ids.shape[1]:].tolist()
        text = tok.decode(new_ids, skip_special_tokens=True)
        m = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.S)
        patch = m.group(1) if m else text
        sm.fire(Event.PATCH_EMITTED)
        r = reflex_check(patch, allowlist=allow)
        ok = r.ok and "compare_digest" in patch      # stand-in for running the test suite
        print(f"attempt k={sm.k + 1}: reflex_ok={r.ok} fixed={ok} mask_stats={proc.stats}")
        if ok:
            sm.fire(Event.BUILD_PASSED); sm.fire(Event.TESTS_PASSED)
        else:
            trie.insert(new_ids)
            sm.fire(Event.BUILD_FAILED, payload={"error": r.error or "test failed"})
            messages = messages[:1] + [{"role": "assistant", "content": text},
                                       {"role": "user", "content": f"That failed: {r.error or 'tests failed'}. Produce a different fix."}]
    print("Final state:", sm.current.phase)
    print(patch)

## 6. Benchmark & Loop 3 calibration

Build the task file from SWE-bench Lite, run the full ablation (Section 6.3 of the README), then calibrate:

```bash
python experiments/prepare_tasks.py --repos-dir /data/swe --out experiments/tasks.jsonl
python experiments/run_benchmark.py --tasks experiments/tasks.jsonl \
    --configs baseline baseline_k3 reflex_only static_mask full \
    --trace-dir experiments/traces --ablation-csv experiments/results_ablation.csv
python -m src.state_machine experiments/traces
```

The cell below calibrates over the statechart trace from Section 3.

In [ ]:
from src import calibrate_from_traces, apply_calibration
trace_file = pathlib.Path(tempfile.mkdtemp()) / "demo_trace.jsonl"
sm.dump_trace(trace_file)
cal = calibrate_from_traces(trace_file)
print(cal)
apply_calibration(cfg, cal)["governance"]


In [ ]:
import pandas as pd
# Header only until a real run writes it (run_benchmark.py --ablation-csv ...).
# The README's Table 6.3 figures are hypothesised targets, not measurements.
df = pd.read_csv(os.path.join(ROOT, "experiments", "results_ablation.csv"))
df if len(df) else print("No measured results yet: results_ablation.csv is header-only.")